# Quantitative Analysis: Facial Recognition & Down Syndrome Audit

**Eticas Foundation** | Adversarial Audit  
**Published:** August 2023  
**Systems audited:** Azul (Zurich Insurance Group) + DeepFace framework  

This notebook reproduces the quantitative findings from the audit report *Invisible No More: The Impact of Facial Recognition on People with Disabilities* (Eticas, August 2023). It covers:

**Part 1 — Azul System (live testing, N=40)**
- Age prediction error by group and gender
- BMI prediction error by group and gender
- Sample composition

**Part 2 — DeepFace Framework (image datasets, N=120)**
- Gender classification accuracy and recall (DS vs NoDS)
- Gender classification by ethnicity (intersectional)
- Ethnicity classification accuracy (DS vs NoDS)
- Ethnicity classification by gender (intersectional)
- Emotion classification
- Age prediction MAE
- Fairness metrics (false positive rate balance)

---
> ⚠️ **Data note:** All values are reproduced from published aggregate statistics. No raw images or individual participant data are included. See `data/README.md` for full provenance and limitations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import warnings
warnings.filterwarnings('ignore')

COLORS = {
    'DS': '#e94560',
    'NoDS': '#1a1a2e',
    'accent': '#4361ee',
    'warn': '#e07b39',
    'muted': '#a8a8b3',
    'light': '#0f3460',
}

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})

azul = pd.read_csv('../data/azul-aggregate-results.csv')
df_comp = pd.read_csv('../data/deepface-dataset-composition.csv')
df_res = pd.read_csv('../data/deepface-results.csv')

print("Datasets loaded.")
print(f"  Azul results: {len(azul)} rows")
print(f"  DeepFace composition: {len(df_comp)} rows")
print(f"  DeepFace results: {len(df_res)} rows")

---
## Part 1: Azul System Audit

In [ ]:
# Sample composition
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Azul Testing — Sample Composition (N=40)', fontweight='bold', fontsize=13)

groups = ['DS', 'NoDS']
male = [12, 9]
female = [8, 11]
smoker = [1, 8]
non_smoker = [19, 12]

x = np.arange(2)
axes[0].bar(x - 0.2, male, 0.35, label='Male', color=COLORS['light'])
axes[0].bar(x + 0.2, female, 0.35, label='Female', color=COLORS['DS'])
axes[0].set_xticks(x); axes[0].set_xticklabels(groups)
axes[0].set_ylabel('Participants'); axes[0].set_title('Gender Distribution')
axes[0].legend(); axes[0].set_ylim(0, 16)
for i, (m, f) in enumerate(zip(male, female)):
    axes[0].text(i-0.2, m+0.2, str(m), ha='center', fontsize=10)
    axes[0].text(i+0.2, f+0.2, str(f), ha='center', fontsize=10)

axes[1].bar(x - 0.2, smoker, 0.35, label='Smoker', color=COLORS['warn'])
axes[1].bar(x + 0.2, non_smoker, 0.35, label='Non-smoker', color=COLORS['accent'])
axes[1].set_xticks(x); axes[1].set_xticklabels(groups)
axes[1].set_ylabel('Participants'); axes[1].set_title('Smoking Status')
axes[1].legend(); axes[1].set_ylim(0, 24)
for i, (s, ns) in enumerate(zip(smoker, non_smoker)):
    axes[1].text(i-0.2, s+0.2, str(s), ha='center', fontsize=10)
    axes[1].text(i+0.2, ns+0.2, str(ns), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('fig1_azul_sample.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Age prediction error comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Azul: Age & BMI Prediction Error — DS vs NoDS', fontweight='bold', fontsize=13)

# Age MAE
groups = ['DS', 'NoDS']
age_mae = [7.19, 4.45]
bars = axes[0].bar(groups, age_mae, color=[COLORS['DS'], COLORS['NoDS']], width=0.4, edgecolor='white')
axes[0].set_ylabel('Mean Absolute Error (years)')
axes[0].set_title('Age Prediction MAE')
for bar, val in zip(bars, age_mae):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'±{val} yrs', ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylim(0, 10)
axes[0].axhline(0, color='grey', linewidth=0.5)

# Deviation ranges
ds_range = [-14, 21]
nods_range = [-9, 18]
for i, (label, r, color) in enumerate(zip(groups, [ds_range, nods_range], [COLORS['DS'], COLORS['NoDS']])):
    axes[1].barh(i, r[1], left=0, height=0.4, color=color, alpha=0.7, label=f'{label}: {r[0]} to +{r[1]} yrs')
    axes[1].barh(i, r[0], left=0, height=0.4, color=color, alpha=0.7)
    axes[1].text(r[1] + 0.3, i, f'+{r[1]}', va='center', fontsize=9)
    axes[1].text(r[0] - 0.3, i, f'{r[0]}', va='center', ha='right', fontsize=9)
axes[1].axvline(0, color='black', linewidth=1)
axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(groups)
axes[1].set_xlabel('Deviation from actual age (years)')
axes[1].set_title('Age Prediction Deviation Range\n(negative = underestimation, positive = overestimation)')
axes[1].set_xlim(-20, 28)

plt.tight_layout()
plt.savefig('fig2_azul_age_error.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key finding: DS participants show 62% higher MAE than NoDS (7.19 vs 4.45 years)")
print("DS group shows systematic overestimation overall, but severe underestimation for women specifically")

In [ ]:
# Gender bias in age and BMI prediction (Azul)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Azul: Gender Bias in DS Group — Selected Cases', fontweight='bold', fontsize=13)

# Age gender bias — illustrative cases from report
cases_age = {
    'Woman A (actual 24)': (24, 8),
    'Woman B (actual 23)': (23, 5),
    'Man X (actual 28)': (28, 33),
    'Man Y (actual 21)': (21, 33),
}
labels = list(cases_age.keys())
actuals = [v[0] for v in cases_age.values()]
predicted = [v[1] for v in cases_age.values()]
colors_age = [COLORS['DS'], COLORS['DS'], COLORS['light'], COLORS['light']]

x = np.arange(len(labels))
axes[0].bar(x - 0.2, actuals, 0.35, label='Actual age', color='#a8a8b3')
axes[0].bar(x + 0.2, predicted, 0.35, label='Azul predicted', color=colors_age)
for i, (a, p) in enumerate(zip(actuals, predicted)):
    diff = p - a
    sign = '+' if diff > 0 else ''
    axes[0].text(i, max(a, p) + 0.8, f'{sign}{diff}', ha='center', fontsize=9,
                 color=COLORS['DS'] if diff < 0 else COLORS['warn'], fontweight='bold')
axes[0].set_xticks(x); axes[0].set_xticklabels(labels, rotation=15, ha='right', fontsize=8)
axes[0].set_ylabel('Age (years)'); axes[0].set_title('Age Prediction — Illustrative Cases\n(DS group, red=women, blue=men)')
axes[0].legend(); axes[0].set_ylim(0, 42)

# BMI gender bias — illustrative cases
cases_bmi = {
    'Woman 1 (actual 26.6)': (26.6, 31.8),
    'Woman 2 (actual 23.7)': (23.7, 22.6),
    'Man 1 (actual 37.9)': (37.9, 30.4),
    'Man 2 (actual 30.9)': (30.9, 29.8),
}
labels_bmi = list(cases_bmi.keys())
actuals_bmi = [v[0] for v in cases_bmi.values()]
predicted_bmi = [v[1] for v in cases_bmi.values()]

x2 = np.arange(len(labels_bmi))
axes[1].bar(x2 - 0.2, actuals_bmi, 0.35, label='Actual BMI', color='#a8a8b3')
axes[1].bar(x2 + 0.2, predicted_bmi, 0.35, label='Azul predicted', color=colors_age)
for i, (a, p) in enumerate(zip(actuals_bmi, predicted_bmi)):
    diff = round(p - a, 1)
    sign = '+' if diff > 0 else ''
    axes[1].text(i, max(a, p) + 0.3, f'{sign}{diff}', ha='center', fontsize=9,
                 color=COLORS['DS'] if diff > 0 else COLORS['warn'], fontweight='bold')
axes[1].set_xticks(x2); axes[1].set_xticklabels(labels_bmi, rotation=15, ha='right', fontsize=8)
axes[1].set_ylabel('BMI (kg/m²)'); axes[1].set_title('BMI Prediction — Illustrative Cases\n(DS group, red=women, blue=men)')
axes[1].legend(); axes[1].set_ylim(0, 42)

plt.tight_layout()
plt.savefig('fig3_azul_gender_bias.png', dpi=150, bbox_inches='tight')
plt.show()
print("Age MAE summary: DS=7.19 yrs | NoDS=4.45 yrs")
print("BMI MAE summary: DS=3.82 kg/m² | NoDS=2.98 kg/m²")

---
## Part 2: DeepFace Framework Analysis

In [ ]:
# Dataset composition
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('DeepFace Test Datasets — Composition (N=120 images)', fontweight='bold', fontsize=13)

eth_order = ['Asian', 'Black', 'Indian', 'Latino_Hispanic', 'Middle_Eastern', 'White']
eth_colors = ['#e94560', '#1a1a2e', '#4361ee', '#e07b39', '#7b2d8b', '#0f3460']

# Gender
axes[0].bar(['DS', 'NoDS'], [30, 30], label='Male', color=COLORS['light'], width=0.4)
axes[0].bar(['DS', 'NoDS'], [30, 30], bottom=[30, 30], label='Female', color=COLORS['DS'], width=0.4)
axes[0].set_title('Gender (balanced)'); axes[0].set_ylabel('Images')
axes[0].legend(); axes[0].set_ylim(0, 75)
axes[0].text(0, 65, '30M / 30F', ha='center', fontsize=9)
axes[0].text(1, 65, '30M / 30F', ha='center', fontsize=9)

# Ethnicity
for i, (eth, col) in enumerate(zip(eth_order, eth_colors)):
    axes[1].bar(['DS', 'NoDS'], [10, 10], bottom=[i*10, i*10], color=col, label=eth.replace('_', ' '))
axes[1].set_title('Ethnicity (balanced, 10 per group)'); axes[1].set_ylabel('Images')
axes[1].legend(fontsize=7, loc='upper right')

# Emotion
ds_em = [35, 25]; nods_em = [32, 28]
axes[2].bar(['DS', 'NoDS'], [ds_em[0], nods_em[0]], label='Happy', color=COLORS['accent'], width=0.4)
axes[2].bar(['DS', 'NoDS'], [ds_em[1], nods_em[1]], bottom=[ds_em[0], nods_em[0]], label='Neutral', color=COLORS['muted'], width=0.4)
axes[2].set_title('Emotion Expression'); axes[2].set_ylabel('Images')
axes[2].legend(); axes[2].set_ylim(0, 75)

plt.tight_layout()
plt.savefig('fig4_deepface_composition.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Gender classification — overall and by sex
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('DeepFace: Gender Classification Performance', fontweight='bold', fontsize=13)

# Overall accuracy
acc = [0.717, 0.974]
benchmark = 0.9744
bars = axes[0].bar(['DS', 'NoDS'], acc, color=[COLORS['DS'], COLORS['NoDS']], width=0.4)
axes[0].axhline(benchmark, color=COLORS['warn'], linestyle='--', linewidth=1.5, label=f'Reported benchmark: {benchmark}')
axes[0].set_ylabel('Accuracy'); axes[0].set_title('Overall Gender Accuracy')
axes[0].set_ylim(0, 1.1)
axes[0].legend(fontsize=9)
for bar, val in zip(bars, acc):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.1%}', ha='center', fontsize=11, fontweight='bold')

# Recall by sex
categories = ['DS Male', 'DS Female', 'NoDS Male', 'NoDS Female']
recalls = [1.0, 0.433, 1.0, 0.800]
bar_colors = [COLORS['light'], COLORS['DS'], COLORS['light'], COLORS['DS']]
bars2 = axes[1].bar(categories, recalls, color=bar_colors, edgecolor='white')
axes[1].set_ylabel('Recall'); axes[1].set_title('Recall by Sex — DS vs NoDS')
axes[1].set_ylim(0, 1.15)
for bar, val in zip(bars2, recalls):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.1%}', ha='center', fontsize=10, fontweight='bold')
plt.xticks(rotation=15, ha='right')

plt.tight_layout()
plt.savefig('fig5_deepface_gender.png', dpi=150, bbox_inches='tight')
plt.show()
print("Key finding: 43.3% recall for DS women vs 100% for all men and 80% for NoDS women")

In [ ]:
# Intersectional: gender accuracy by ethnicity
ethnicities = ['Asian', 'Black', 'Indian', 'Latino\nHispanic', 'Middle\nEastern', 'White']
ds_female_recall = [0.000, 0.250, 0.250, 0.500, 0.500, 0.833]
nods_female_recall = [0.800, 0.800, 0.800, 0.400, 1.000, 1.000]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(ethnicities))
bars1 = ax.bar(x - 0.2, ds_female_recall, 0.35, label='DS women', color=COLORS['DS'])
bars2 = ax.bar(x + 0.2, nods_female_recall, 0.35, label='NoDS women', color=COLORS['NoDS'])
ax.axhline(1.0, color=COLORS['light'], linestyle=':', linewidth=1, alpha=0.5, label='Men (all groups): 1.0')
ax.set_xticks(x); ax.set_xticklabels(ethnicities)
ax.set_ylabel('Female recall (correctly classified as female)')
ax.set_title('Intersectional Gender Classification: Female Recall by Ethnicity', fontweight='bold')
ax.set_ylim(0, 1.15)
ax.legend()
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.0%}', ha='center', fontsize=8, color=COLORS['DS'])
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.0%}', ha='center', fontsize=8, color=COLORS['NoDS'])

plt.tight_layout()
plt.savefig('fig6_deepface_gender_x_ethnicity.png', dpi=150, bbox_inches='tight')
plt.show()
print("Key finding: Asian DS women — 0% recall. White DS women — highest recall at 83%.")

In [ ]:
# Ethnicity classification accuracy
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('DeepFace: Ethnicity Classification Performance', fontweight='bold', fontsize=13)

# Overall accuracy
eth_acc = [0.383, 0.667]
benchmark_eth = 0.68
bars = axes[0].bar(['DS', 'NoDS'], eth_acc, color=[COLORS['DS'], COLORS['NoDS']], width=0.4)
axes[0].axhline(benchmark_eth, color=COLORS['warn'], linestyle='--', linewidth=1.5, label=f'Reported benchmark: {benchmark_eth}')
axes[0].set_ylabel('Accuracy'); axes[0].set_title('Overall Ethnicity Accuracy')
axes[0].set_ylim(0, 1.0); axes[0].legend(fontsize=9)
for bar, val in zip(bars, eth_acc):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.1%}', ha='center', fontsize=11, fontweight='bold')

# Ethnicity accuracy by group × gender
eth_labels = ['Asian', 'Black', 'Indian', 'Latino\nHispanic', 'Middle\nEastern', 'White']
ds_male_eth = [0.833, 0.667, 0.167, 0.250, 0.000, 0.500]
ds_female_eth = [0.250, 0.500, 0.000, 0.500, 0.167, 0.500]
nods_male_eth = [1.000, 1.000, 0.400, 0.200, 0.800, 1.000]
nods_female_eth = [1.000, 0.800, 0.200, 0.400, 0.200, 1.000]

x = np.arange(len(eth_labels))
axes[1].plot(x, ds_male_eth, 'o-', color=COLORS['light'], linewidth=2, label='DS male', markersize=6)
axes[1].plot(x, ds_female_eth, 's--', color=COLORS['DS'], linewidth=2, label='DS female', markersize=6)
axes[1].plot(x, nods_male_eth, 'o-', color='#4361ee', linewidth=2, label='NoDS male', markersize=6)
axes[1].plot(x, nods_female_eth, 's--', color='#7b2d8b', linewidth=2, label='NoDS female', markersize=6)
axes[1].set_xticks(x); axes[1].set_xticklabels(eth_labels, fontsize=8)
axes[1].set_ylabel('Correct ethnicity classification rate')
axes[1].set_title('Ethnicity Recall by Subgroup')
axes[1].set_ylim(-0.05, 1.15); axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig7_deepface_ethnicity.png', dpi=150, bbox_inches='tight')
plt.show()
print("Key finding: DS overall 38.3% vs NoDS 66.7%. Indian and Middle Eastern DS subgroups worst performers.")

In [ ]:
# Emotion and age summary
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('DeepFace: Emotion Classification & Age MAE', fontweight='bold', fontsize=13)

# Emotion accuracy + confidence
em_acc = [0.567, 0.583]
em_conf = [8.052, 13.193]
ax1_twin = axes[0].twinx()
b1 = axes[0].bar(['DS', 'NoDS'], em_acc, color=[COLORS['DS'], COLORS['NoDS']], width=0.35, alpha=0.8, label='Accuracy')
l1 = ax1_twin.plot(['DS', 'NoDS'], em_conf, 'D--', color=COLORS['warn'], linewidth=2, markersize=8, label='Mean confidence (true label)')
axes[0].set_ylabel('Accuracy'); ax1_twin.set_ylabel('Mean confidence score')
axes[0].set_title('Emotion Classification\n(similar accuracy, but lower confidence for DS)')
axes[0].set_ylim(0, 1); ax1_twin.set_ylim(0, 18)
for bar, val in zip(b1, em_acc):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.1%}', ha='center', fontsize=10, fontweight='bold')
lines = [mpatches.Patch(color=COLORS['DS'], label='DS accuracy'),
         mpatches.Patch(color=COLORS['NoDS'], label='NoDS accuracy'),
         plt.Line2D([0], [0], color=COLORS['warn'], marker='D', linestyle='--', label='Mean confidence')]
axes[0].legend(handles=lines, fontsize=8)

# Age MAE comparison
age_mae_vals = [10.583, 9.167, 4.65]
age_labels = ['DS\n(our test)', 'NoDS\n(our test)', 'Reported\nbenchmark']
age_colors = [COLORS['DS'], COLORS['NoDS'], COLORS['warn']]
bars = axes[1].bar(age_labels, age_mae_vals, color=age_colors, width=0.4)
axes[1].set_ylabel('MAE (years)'); axes[1].set_title('Age Prediction MAE — DeepFace')
axes[1].set_ylim(0, 14)
for bar, val in zip(bars, age_mae_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'±{val}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('fig8_deepface_emotion_age.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary table of all key metrics
summary = pd.DataFrame({
    'System': ['Azul', 'Azul', 'DeepFace', 'DeepFace', 'DeepFace', 'DeepFace', 'DeepFace', 'DeepFace'],
    'Metric': [
        'Age MAE (years)', 'BMI MAE (kg/m²)',
        'Gender accuracy', 'Gender recall — women',
        'Ethnicity accuracy', 'Emotion accuracy',
        'Age MAE (years)', 'Age MAE (years)'
    ],
    'Group': ['DS', 'DS', 'DS', 'DS', 'DS', 'DS', 'DS', 'NoDS'],
    'Value_DS_or_noted': [7.19, 3.82, 0.717, 0.433, 0.383, 0.567, 10.583, 9.167],
    'Value_NoDS_or_benchmark': [4.45, 2.98, 0.974, 0.800, 0.667, 0.583, '±4.65 (benchmark)', '±4.65 (benchmark)'],
    'Disparity': [
        '+61.6% higher error', '+28.2% higher error',
        '-26.5% lower accuracy', '-46.7% lower recall',
        '-42.6% lower accuracy', 'Similar',
        '+128% higher error', '+97% higher error'
    ]
})
print("=== AUDIT FINDINGS SUMMARY ===")
print(summary.to_string(index=False))

---
## Methodology Note

**Azul testing:** 40 participants (20 DS from Cedown Jerez; 20 NoDS control) each ran the Azul virtual insurance assessment. Researchers observed and recorded the system's outputs for age, BMI, and smoking status, comparing against actual participant characteristics. Error metrics are mean absolute differences between predicted and actual values.

**DeepFace testing:** Two image datasets (60 DS images, 60 NoDS images) were compiled from the internet, balanced by gender (30M/30F) and ethnicity (10 per 6 categories). Labels were assigned manually (gender, emotion) or from databases (age, ethnicity). The DeepFace framework (Serengil & Ozpinar) was run across all images. Performance was evaluated using accuracy, precision, recall, F1-score (gender/emotion/ethnicity) and MAE (age). Fairness was assessed via false positive rate balance and conditional probability analysis across gender × ethnicity and gender × emotion subgroups.

**Note on equalized odds:** The analysis applies an equalized odds fairness criterion — equal true positive and false positive rates across demographic groups. Results show this criterion is violated for gender classification (DS women vs NoDS women) and ethnicity classification (DS vs NoDS across all subgroups).